# Orbital Debris Database: Exploring the Landscape

**Database:** orbital_debris.db (SQLite)  
**Docs:** See quick reports in notebook 04 and data dictionary in `docs/schema` for full schema and field definitions.

**Goal:** Surface patterns, outliers, and new questions in the orbital debris database to guide deeper analysis and visualization.

### Why this notebook?
This notebook builds on the cleaned, joined database. Instead of repeating full audits, it focuses on targeted exploration, query prototyping, and visual storytelling to support research questions and analysis.

### What we do here
1. **Prototype questions:** Use SQL and pandas to dig into specific trends, outliers, or hypotheses.
2. **Visualize:** Create quick charts to spot patterns and inform next steps.
3. **Document findings:** Save useful queries, charts, and notes for future reference.

*For table structure, nulls, and sample data, see the end of notebook 04 and the data dictionary.*

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import sqlite3 as sql
import utility as utils
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import display, Markdown

plt.rcParams.update({
    "grid.alpha": 0.3,
    "axes.facecolor": "#333333",
    "figure.facecolor": "#333333",
    "text.color": "white",
    "axes.labelcolor": "white",
    "xtick.color": "white",
    "ytick.color": "white",
    "legend.facecolor": "#333333",
    "legend.edgecolor": "white"
})
 
orbital_debris_conn = sql.connect('../data/clean/orbital_debris.db')
print(plt.colormaps())

**Note:** Full quick reports for all database tables are generated at the end of notebook 04 (`04_orbital_debris_synthesis.ipynb`). See that notebook for detailed table structure, nulls, and sample data. For field definitions and schema details, refer to the data dictionary in `docs/schema`. This notebook focuses on exploration and analysis.

### Launch Year Trends: Decoupling Point in Orbital Growth
 
**Intent:** Identify when orbital object growth shifted from linear to exponential by visualizing launches per year.
 
**Method:**
- Query the satellites table for launch years.
- Count and plot the number of objects launched per year.
- Visually inspect for inflection points or rapid growth phases.

In [ ]:
# Query: Launches per year
launches_per_year_query = """
SELECT
    satellites.norad_id,
    satellites.object_name,
    launch_events.launch_date
FROM satellites
JOIN launch_events ON satellites.launch_id = launch_events.launch_id
WHERE launch_events.launch_date IS NOT NULL
ORDER BY launch_events.launch_date ASC;
"""

# Run the query and store the result
df_launches_per_year = pd.read_sql_query(launches_per_year_query, orbital_debris_conn)

# Convert to datetime and extract year
df_launches_per_year['launch_year'] = pd.to_datetime(df_launches_per_year['launch_date'], errors='coerce').dt.year

df_launches_per_year.to_parquet('../data/clean/results/launches_per_year_exp.parquet', index=False)

In [ ]:
# Validation: Check for nulls, unique years, and overall data integrity
print('Row count:', len(df_launches_per_year))
display(df_launches_per_year.head())
display(df_launches_per_year.tail())

print('Unique years:', df_launches_per_year["launch_year"].nunique())
print('Year range:', df_launches_per_year["launch_year"].min(), '-', df_launches_per_year["launch_year"].max())

print('Nulls per column:')
display(df_launches_per_year.isnull().sum())

print('Sample years:', df_launches_per_year["launch_year"].unique()[:10])

In [ ]:
# Count the number of launches per year
launch_counts = df_launches_per_year['launch_year'].value_counts().sort_index()

plt.figure(figsize=(16,8))
launch_counts.plot(kind='bar', color='skyblue')

plt.title('Number of Objects Launched per Year')
plt.xlabel('Year')
plt.ylabel('Number of Objects')
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../charts/questions/exploratory/svg/launches_per_year_exp.svg', bbox_inches='tight')
plt.savefig('../charts/questions/exploratory/png/launches_per_year_exp.png', bbox_inches='tight', dpi=300)
plt.show()

### Debris Growth Over Time: Annual Debris Object Counts

**Intent:** Visualize the trend of debris objects in orbit by year to identify growth patterns and inflection points.

**Method:**
- Query the debris table (or filter satellites table for debris type) for object type = 'DEBRIS'.
- Extract/convert launch or identification year.
- Count and plot the number of debris objects per year.
- Validate row/column counts and check for missing years or anomalies.

In [ ]:
# Query: Debris objects per year
debris_query = """
SELECT
    satellites.norad_id,
    satellites.object_name,
    launch_events.launch_date
FROM satellites
JOIN launch_events ON satellites.launch_id = launch_events.launch_id
WHERE satellites.object_type = 'DEBRIS' AND launch_events.launch_date IS NOT NULL
ORDER BY launch_events.launch_date ASC;
"""

# Run the query and store the result
df_debris_per_year = pd.read_sql_query(debris_query, orbital_debris_conn)

# Convert to datetime and extract year
df_debris_per_year['debris_year'] = pd.to_datetime(df_debris_per_year['launch_date'], errors='coerce').dt.year

# Save the query result to a parquet file for future use.
df_debris_per_year.to_parquet('../data/clean/results/debris_per_year_exp.parquet', index=False)

In [ ]:
# Validation: Debris Growth Data Integrity
print('Row count:', len(df_debris_per_year))
display(df_debris_per_year.head())
display(df_debris_per_year.tail())

print('Unique years:', df_debris_per_year["debris_year"].nunique())
print('Year range:', df_debris_per_year["debris_year"].min(), '-', df_debris_per_year["debris_year"].max())

print('Nulls per column:')
display(df_debris_per_year.isnull().sum())

print('Sample years:', df_debris_per_year["debris_year"].unique()[:10])

In [ ]:
# Count debris objects per year
debris_counts = df_debris_per_year['debris_year'].value_counts().sort_index()

plt.figure(figsize=(16,8))

debris_counts.plot(kind='bar', color='orange')

plt.title('Number of Debris Objects Added per Year')
plt.xlabel('Year')
plt.ylabel('Number of Debris Objects')
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../charts/questions/exploratory/svg/debris_per_year_exp.svg', bbox_inches='tight')
plt.savefig('../charts/questions/exploratory/png/debris_per_year_exp.png', bbox_inches='tight', dpi=300)
plt.show()

### Object Type Distribution: How the Mix of Payloads, Rocket Bodies, and Debris Changes Over Time

**Intent:** Show how the proportions of payloads, rocket bodies, and debris have shifted over time, highlighting changes in the orbital environment.

**Method:**
- Query the satellites table (joined with launch_events) for object type and launch year.
- Group and count by year and object type.
- Visualize as a stacked bar or area chart to show the mix over time.
- Validate for missing years, nulls, and object type coverage.

In [ ]:
# Query: Object type distribution by year
object_type_query = """
SELECT
    satellites.norad_id,
    satellites.object_type,
    launch_events.launch_date
FROM satellites
JOIN launch_events ON satellites.launch_id = launch_events.launch_id
WHERE launch_events.launch_date IS NOT NULL
ORDER BY launch_events.launch_date ASC;
"""

# Run the query and store the result
df_type_year = pd.read_sql_query(object_type_query, orbital_debris_conn)

# Convert to datetime and extract year
df_type_year['launch_year'] = pd.to_datetime(df_type_year['launch_date'], errors='coerce').dt.year

# Save the query result to a parquet file for future use.
df_type_year.to_parquet('../data/clean/results/object_type_year_exp.parquet', index=False)


In [ ]:
# Validation: Object Type Distribution Data Integrity
print('Row count:', len(df_type_year))
display(df_type_year.head())

print('Unique years:', df_type_year["launch_year"].nunique())
print('Year range:', df_type_year["launch_year"].min(), '-', df_type_year["launch_year"].max())
print('Object types:', df_type_year["object_type"].unique())

print('Nulls per column:')
display(df_type_year.isnull().sum())

print('Sample years:', df_type_year["launch_year"].unique()[:10])

In [ ]:
# Group by year and object type, count
type_counts = df_type_year.groupby(['launch_year', 'object_type']).size().unstack(fill_value=0)

# Plot: Stacked area chart
plt.figure(figsize=(16,10))
type_counts.plot.area(ax=plt.gca(), alpha=0.85, colormap='tab20c')

plt.title('Object Type Distribution Over Time (Area Chart)')
plt.xlabel('Year')
plt.ylabel('Number of Objects')
plt.grid(axis='y', alpha=0.3)
plt.legend(title='Object Type')

plt.tight_layout()
plt.savefig('../charts/questions/exploratory/svg/object_type_distribution_exp.svg', bbox_inches='tight')
plt.savefig('../charts/questions/exploratory/png/object_type_distribution_exp.png', bbox_inches='tight', dpi=300)
plt.show()

# vertical bar chart alternative

plt.figure(figsize=(16,10))
type_counts.plot(kind='bar', stacked=True, ax=plt.gca(), alpha=0.85, colormap='tab20c')

plt.title('Object Type Distribution Over Time (Bar Chart)')
plt.xlabel('Year')
plt.ylabel('Number of Objects')
plt.grid(axis='y', alpha=0.3)
plt.legend(title='Object Type')

plt.tight_layout()
plt.savefig('../charts/questions/exploratory/svg/object_type_distribution_bar_exp.svg', bbox_inches='tight')
plt.savefig('../charts/questions/exploratory/png/object_type_distribution_bar_exp.png', bbox_inches='tight', dpi=300)
plt.show()


### High-Risk Object Counts: Number and Share Above Thresholds

**Intent:** Quantify the number and proportion of objects considered high-risk based on velocity and/or kinetic energy.

**Method:**
- Define velocity and/or kinetic energy thresholds for high risk.
- Query the database for objects above these thresholds.
- Calculate counts and share of total population.
- Validate for nulls and edge cases.

In [ ]:
velocity_threshold = 7800  # m/s
kinetic_energy_threshold = 1e10  # joules

# Query: High-risk objects by velocity or kinetic energy
# We will be adding a row called 'altitude_km' which ( apogee + perigee ) /  2, 
# using the mean of perigee and apogee to estimate an object’s “typical” or “representative” 
# orbital altitude is standard practice in both academic literature and NASA/ESA data analysis.

high_risk_query = f'''
SELECT
  satellites.norad_id,
  satellites.object_name,
  satellites.object_type,
  launch_events.launch_date,
  orbital_data.orbit_class,
  orbital_data.apogee_km,
  orbital_data.perigee_km,
  orbital_data.proxy_mass_kg,
  (orbital_data.apogee_km + orbital_data.perigee_km) / 2 AS altitude_km,
  risk_assessment.velocity_kms,
  risk_assessment.kinetic_joules
FROM satellites 
JOIN risk_assessment ON satellites.norad_id = risk_assessment.norad_id 
JOIN orbital_data ON satellites.norad_id = orbital_data.norad_id
JOIN launch_events ON satellites.launch_id = launch_events.launch_id
WHERE risk_assessment.kinetic_joules IS NOT NULL;
'''

df_all_sats = pd.read_sql_query(high_risk_query, orbital_debris_conn)
df_all_sats['risk_category'] = df_all_sats.apply(utils.classify_risk_category, axis=1)

df_high_risk = df_all_sats[
  (df_all_sats['risk_category'] == 'Extremely High Risk')
].copy()

df_all_sats['launch_year'] = pd.to_datetime(df_all_sats['launch_date'], errors='coerce').dt.year
df_high_risk['launch_year'] = pd.to_datetime(df_high_risk['launch_date'], errors='coerce').dt.year

# Save the query result to a parquet file for future use.
df_all_sats.to_parquet('../data/clean/results/all_sats_with_risk_exp.parquet', index=False)
df_all_sats.to_parquet('../data/clean/results/high_risk_sats_exp.parquet', index=False)

In [ ]:
# Count and share
total_objects = pd.read_sql_query('SELECT COUNT(*) as n FROM satellites;', orbital_debris_conn)['n'][0]
high_risk_count = len(df_high_risk)
high_risk_share = high_risk_count / total_objects if total_objects else float('nan')

print(f'Total objects with risk data: {len(df_all_sats)}')
display(df_all_sats.head())

print(f'High-risk objects: {high_risk_count} of {total_objects} ({high_risk_share:.2%})')
display(df_high_risk.head())

# Validation
print('Nulls per column:')
display(df_high_risk.isnull().sum())

### Orbit Class Breakdown: High-Risk Objects by Orbit Class

**Intent:** Show how high-risk objects are distributed across orbit classes (LEO, MEO, GEO, etc.).

**Method:**
- Use the high-risk subset from above.
- Group and count by orbit class.
- Visualize as a bar chart or pie chart.
- Validate for missing/unknown classes.

In [ ]:
# Group high-risk objects by orbit class
if 'orbit_class' in df_high_risk.columns:
    orbit_counts = df_high_risk['orbit_class'].value_counts().sort_values(ascending=False)
    
    plt.figure(figsize=(10,6))
    orbit_counts.plot(kind='bar', colormap='Set1')
    
    plt.title('High-Risk Objects by Orbit Class')
    plt.xlabel('Orbit Class')
    plt.ylabel('Number of High-Risk Objects')
    plt.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../charts/questions/exploratory/svg/hr_orbit_class_exp.svg', bbox_inches='tight')
    plt.savefig('../charts/questions/exploratory/png/hr_orbit_class_exp.png', bbox_inches='tight', dpi=300)  
    plt.show()
    
    print(orbit_counts)
else:
    print('orbit_class column not found in high-risk data.')

### Altitude Band Analysis: 400–600 km LEO High-Risk Objects

**Intent:** Focus on high-risk objects in the 400–600 km altitude band (LEO), a region of particular congestion and risk.

**Method:**
- Filter high-risk objects for those with altitude between 400 and 600 km.
- Plot histogram or density plot of their altitudes.
- Validate for nulls and outliers.

In [ ]:
# Filter for 400–600 km LEO high-risk objects
if 'altitude_km' in df_high_risk.columns:
    leo_band = df_high_risk[(df_high_risk['altitude_km'] >= 400) & (df_high_risk['altitude_km'] <= 600)]
    
    plt.figure(figsize=(10,6))
    sns.histplot(leo_band['altitude_km'], bins=30, kde=True, color='gold')
    
    plt.title('High-Risk Objects in 400–600 km LEO Band')
    plt.xlabel('Altitude (km)')
    plt.ylabel('Count')
    plt.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../charts/questions/exploratory/svg/high_risk_banded_exp.svg', bbox_inches='tight')
    plt.savefig('../charts/questions/exploratory/png/high_risk_banded_exp.png', bbox_inches='tight', dpi=300)
    plt.show()
    
    print(f'Objects in 400–600 km band: {len(leo_band)}')
else:
    print('altitude column not found in high-risk data.')

### Mass Distribution: High-Risk Objects (if available)

**Intent:** Examine the mass distribution of high-risk objects to identify heavy outliers or trends.

**Method:**
- If mass data is available, plot histogram or boxplot for high-risk objects.
- Validate for nulls and extreme values.

In [ ]:
# Mass distribution for high-risk objects: scatter (strip) plot only
if 'proxy_mass_kg' in df_high_risk.columns:
    mass_nonzero = df_high_risk['proxy_mass_kg'].dropna()
    mass_nonzero = mass_nonzero[mass_nonzero > 0]
    
    plt.figure(figsize=(12, 6))
    
    # Add jitter to y-axis for better visibility
    y_jitter = np.random.uniform(-0.1, 0.1, size=len(mass_nonzero))
    
    plt.scatter(mass_nonzero, 1 + y_jitter, alpha=0.7, color='lime', edgecolor='black', s=30)
    plt.yticks([])
    plt.xlabel('Mass (kg)')
    plt.title('Mass Distribution (Scatter/Strip Plot) - High-Risk Only')
    plt.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../charts/questions/exploratory/svg/high_risk_mass_distribution.svg', bbox_inches='tight')
    plt.savefig('../charts/questions/exploratory/png/high_risk_mass_distribution.png', bbox_inches='tight', dpi=300)
    plt.show()
    
    print(f'Objects with nonzero mass data: {len(mass_nonzero)}')
else:
    print('proxy_mass_kg column not found in high-risk data.')